# Building a RESTful API with Node.js and Express: A Step-by-Step Tutorial

Welcome to this hands-on tutorial! In the next 1.15 hours, we'll build a complete RESTful API for a simple task manager. We'll cover Node.js, Express, RESTful principles, and CRUD operations. Let's dive in!

## Prerequisites
- Node.js (v14 or higher) and npm installed
- Basic understanding of JavaScript
- A code editor (like VS Code)

## Step 1: Project Setup (5 minutes)

### 1.1 Create the Project Directory

In [ ]:
mkdir june13-api-session
cd june13-api-session

### 1.2 Initialize the Project

In [ ]:
npm init -y

### 1.3 Install Dependencies

In [ ]:
npm install express dotenv cors morgan
npm install --save-dev nodemon jest supertest

### 1.4 Update `package.json`
Add the following scripts:

In [ ]:
%%writefile package.json
{
  "scripts": {
    "start": "node server.js",
    "dev": "nodemon server.js",
    "test": "jest"
  }
}

### 1.5 Create a `.env` File

In [ ]:
%%writefile .env
PORT=3000
NODE_ENV=development

### 1.6 Create a `.gitignore` File

In [ ]:
ss%%writefile .gitignore
node_modules/
.env

## Step 2: Setting Up the Express Server (10 minutes)

### 2.1 Create `server.js`
This is the entry point of our application. We'll set up Express, middleware, and routes.

In [ ]:
%%writefile server.js
import 'dotenv/config';
import express from 'express';
import cors from 'cors';
import morgan from 'morgan';
import { errorHandler } from './middleware/errorHandler.js';
import { logger } from './middleware/logger.js';
import tasksRouter from './routes/tasks.js';

const app = express();
const PORT = process.env.PORT || 3000;

// Middleware
app.use(cors());
app.use(express.json());
app.use(morgan('dev'));
app.use(logger);

// Routes
app.use('/tasks', tasksRouter);

// Error handling
app.use(errorHandler);

// Start server
app.listen(PORT, () => {
  console.log(`Server is running on port ${PORT}`);
});

### 2.2 Key Concepts Introduced:
- **Express**: A fast, unopinionated, minimalist web framework for Node.js.
- **Middleware**: Functions that have access to the request and response objects. They can execute code, modify the request/response, and end the request-response cycle.
- **Environment Variables**: Using `dotenv` to manage configuration.

## Step 3: Creating Middleware (10 minutes)

### 3.1 Error Handling Middleware (`middleware/errorHandler.js`)
This middleware catches errors and sends a consistent JSON response.

In [ ]:
%%writefile middleware/errorHandler.js
export const errorHandler = (err, req, res, next) => {
  console.error(err.stack);
  const status = err.statusCode || 500;
  const message = err.message || 'Something went wrong!';
  res.status(status).json({ success: false, error: message });
};

### 3.2 Logging Middleware (`middleware/logger.js`)
This middleware logs each request's method, URL, status code, and response time.

In [ ]:
%%writefile middleware/logger.js
export const logger = (req, res, next) => {
  const start = Date.now();
  res.on('finish', () => {
    const duration = Date.now() - start;
    console.log(`${req.method} ${req.originalUrl} ${res.statusCode} - ${duration}ms`);
  });
  next();
};

### 3.3 Key Concepts Introduced:
- **Error Handling**: Centralizing error responses for consistency.
- **Logging**: Observability and debugging.

## Step 4: Building the Task Model (10 minutes)

### 4.1 Create `models/taskModel.js`
This file manages an in-memory array of tasks.

In [ ]:
%%writefile models/taskModel.js
let tasks = [];
let nextId = 1;

class Task {
  constructor(text) {
    this.id = nextId++;
    this.text = text;
    this.done = false;
    this.createdAt = new Date();
  }
}

export const taskModel = {
  getAll: () => [...tasks],
  getById: (id) => tasks.find(task => task.id === id),
  create: (text) => {
    const task = new Task(text);
    tasks.push(task);
    return task;
  },
  delete: (id) => {
    const index = tasks.findIndex(task => task.id === id);
    if (index === -1) return false;
    tasks.splice(index, 1);
    return true;
  }
};

### 4.2 Key Concepts Introduced:
- **In-Memory Data Store**: Simulating a database with an array.
- **ES Modules**: Using `export` to share functionality.

## Step 5: Implementing the Task Controller (15 minutes)

### 5.1 Create `controllers/tasksController.js`
This file contains the business logic for handling task-related requests.

In [ ]:
%%writefile controllers/tasksController.js
import { taskModel } from '../models/taskModel.js';

export const tasksController = {
  getTasks: (req, res, next) => {
    try {
      const tasks = taskModel.getAll();
      res.json({ success: true, data: tasks });
    } catch (error) {
      next(error);
    }
  },

  addTask: (req, res, next) => {
    try {
      const { text } = req.body;
      if (!text) {
        return res.status(400).json({ success: false, error: 'Text is required' });
      }
      const task = taskModel.create(text);
      res.status(201).json({ success: true, data: task });
    } catch (error) {
      next(error);
    }
  },

  deleteTask: (req, res, next) => {
    try {
      const id = parseInt(req.params.id);
      const deleted = taskModel.delete(id);
      if (!deleted) {
        return res.status(404).json({ success: false, error: 'Task not found' });
      }
      res.json({ success: true, message: 'Task deleted successfully' });
    } catch (error) {
      next(error);
    }
  }
};

### 5.2 Key Concepts Introduced:
- **Controllers**: Separating business logic from routing.
- **Input Validation**: Checking for required fields.
- **HTTP Status Codes**: Using appropriate status codes (201, 400, 404).

## Step 6: Defining Routes (10 minutes)

### 6.1 Create `routes/tasks.js`
This file defines the API endpoints for tasks.

In [ ]:
%%writefile routes/tasks.js
import express from 'express';
import { tasksController } from '../controllers/tasksController.js';

const router = express.Router();

router.get('/', tasksController.getTasks);
router.post('/', tasksController.addTask);
router.delete('/:id', tasksController.deleteTask);

export default router;